# 117 — Instructor: Type-Safe Structured Extraction with Automatic Retry
## What you'll learn: Pydantic-first LLM output parsing with auto-reask on validation failure
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/117-instructor-extraction/instructor_extraction_workbook.ipynb)

`instructor` is a thin wrapper around any OpenAI-compatible client that adds two things:
1. **Pydantic model validation** — the LLM is forced to return JSON matching your model's schema
2. **Automatic retry (reask)** — if the LLM returns invalid JSON, instructor re-prompts with the validation error, giving the model another chance

The result: `response_model=MyModel` and you get back a validated Python object, not a string.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — how instructor works under the hood |
| 2 | **Setup** — install, API key |
| 3 | **Pydantic models** — MeetingNotes, ProductReview, UserAddress |
| 4 | **Basic extraction** — `client.chat.completions.create(response_model=...)` |
| 5 | **Retry mechanics** — ValidationError → reask → fix |
| 6 | **Field validators** — enforcing business rules on extracted data |
| 7 | **Provider portability** — same code with Anthropic backend |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `instructor`

### Key References
> [instructor GitHub](https://github.com/jxnl/instructor)
>
> [instructor docs](https://python.useinstructor.com/)
>
> [Pydantic field validators](https://docs.pydantic.dev/latest/concepts/validators/)

## Part 1 — Concepts: How Instructor Works

### The raw OpenAI structured output problem

Without instructor, getting structured JSON from an LLM requires:

```python
# Option A: prompt the model to return JSON and hope for the best
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Return a JSON with fields: name, age"}],
)
raw = response.choices[0].message.content  # might not be valid JSON
data = json.loads(raw)  # might fail
# No type checking, no validation, no retry
```

```python
# Option B: use response_format={"type": "json_object"}
# Forces JSON but doesn't enforce schema
# Still need to validate manually
```

### What instructor adds

```python
import instructor
from openai import OpenAI
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int

client = instructor.from_openai(OpenAI())
person = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=Person,       # <-- Pydantic schema
    max_retries=3,               # <-- auto-retry on ValidationError
    messages=[{"role": "user", "content": "John is 30 years old."}]
)
# person is a validated Person instance — not a string, not a dict
print(person.name, person.age)  # John 30
```

### The retry loop

When the LLM returns output that fails Pydantic validation:

```
Attempt 1:
  LLM returns: {"name": "John", "age": "thirty"}  # age is a string
  Pydantic: ValidationError — age must be int
  instructor: reasks with "The response failed validation: age must be int. Try again."

Attempt 2:
  LLM returns: {"name": "John", "age": 30}  # fixed
  Pydantic: OK
  Returns Person(name="John", age=30)
```

Without `max_retries`, one failure raises immediately. With `max_retries=3`, instructor
gives the model 3 chances to produce valid output.

### How instructor passes the schema

Under the hood, instructor converts your Pydantic model to a JSON Schema and passes it as:
- **OpenAI**: `tools=[{"type": "function", "function": {"name": "...", "parameters": schema}}]` with `tool_choice={"type": "function", "function": {"name": "..."}}`
- **Anthropic**: similar tool-use pattern
- **Ollama**: JSON mode with schema in system prompt

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "instructor==1.8.3",
         "openai==1.109.1",
         "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install (using requirements.txt)")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 3 — Pydantic Models for Extraction

### Model design principles

Good extraction models follow these rules:
1. **Field descriptions are prompts** — the `description` in `Field()` is shown to the LLM
2. **Be specific about types** — `list[str]` vs `str` makes a big difference
3. **Use `| None` for optional fields** — don't make the LLM hallucinate missing data
4. **Field validators enforce business rules** — run after the LLM's JSON is parsed

### The three models

```
MeetingNotes:    attendees, decisions, action_items, next_meeting
ProductReview:   sentiment (enum), key_features, score (1-5), would_recommend
UserAddress:     street, city, state (2-letter), zip_code (5-digit)
```

Each uses `Field(description="...")` to guide the LLM's extraction.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, field_validator


class MeetingNotes(BaseModel):
    attendees: list[str] = Field(description="Names of all meeting attendees")
    decisions: list[str] = Field(description="Decisions made during the meeting")
    action_items: list[str] = Field(
        description="Action items with owner and deadline where mentioned"
    )
    next_meeting: str | None = Field(
        default=None,
        description="Date/time of next meeting if mentioned, else null"
    )


class ProductReview(BaseModel):
    sentiment: Literal["positive", "negative", "mixed", "neutral"] = Field(
        description="Overall sentiment: one of positive, negative, mixed, neutral"
    )
    key_features: list[str] = Field(
        description="Product features mentioned (positive or negative)"
    )
    score: float = Field(
        description="Numeric score from 1.0 to 5.0 inferred from review tone",
        ge=1.0,
        le=5.0,
    )
    would_recommend: bool = Field(
        description="Whether the reviewer would recommend this product"
    )


class UserAddress(BaseModel):
    street: str = Field(description="Street number and name")
    city: str = Field(description="City name")
    state: str = Field(description="Two-letter US state code (e.g. CA, NY)")
    zip_code: str = Field(description="5-digit ZIP code")

    @field_validator("state")
    @classmethod
    def normalize_state(cls, v: str) -> str:
        return v.upper().strip()

    @field_validator("zip_code")
    @classmethod
    def validate_zip(cls, v: str) -> str:
        digits = v.strip().replace("-", "")[:5]
        if not digits.isdigit() or len(digits) != 5:
            raise ValueError(f"ZIP code must be 5 digits, got: {v!r}")
        return digits


print("Models defined:")
print(f"  MeetingNotes fields: {list(MeetingNotes.model_fields.keys())}")
print(f"  ProductReview fields: {list(ProductReview.model_fields.keys())}")
print(f"  UserAddress fields: {list(UserAddress.model_fields.keys())}")

# Show the JSON schema instructor will send to the LLM
import json
schema = UserAddress.model_json_schema()
print(f"\nUserAddress JSON schema (sent to LLM):")
print(json.dumps(schema, indent=2))

## Part 4 — Basic Extraction

### Creating the instructor client

```python
import instructor
from openai import OpenAI

client = instructor.from_openai(OpenAI())
```

`instructor.from_openai()` wraps the `openai.OpenAI` client and patches the
`chat.completions.create` method to accept `response_model` and `max_retries`.

Everything else — authentication, model selection, temperature — works the same as vanilla OpenAI.

In [ ]:
import instructor
from openai import OpenAI

client = instructor.from_openai(OpenAI())
print("Instructor client created.")
print(f"Type: {type(client)}")

In [ ]:
# --- Extraction 1: Meeting Notes ---

MEETING_TRANSCRIPT = """
Q3 Planning Meeting — September 15, 2024
Attendees: Sarah Chen (PM), Marcus Webb (Engineering Lead), Priya Nair (Design), Tom Okafor (QA)

After reviewing Q2 metrics, the team decided to push the mobile launch to October 28th
instead of October 15th. Marcus agreed to finalize API endpoints by September 30th.
Priya will deliver design assets by September 25th. Tom is responsible for the automated
test suite by October 10th.

The team also decided to drop the offline mode feature from the initial release.
Sarah will communicate this to stakeholders by end of day Friday.

Next meeting: September 22nd at 10 AM.
"""

print("Extracting MeetingNotes...")
notes = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=MeetingNotes,
    max_retries=3,
    messages=[
        {
            "role": "system",
            "content": "You are an expert meeting notes extractor. Extract structured information accurately.",
        },
        {
            "role": "user",
            "content": f"Extract structured notes from this meeting transcript:\n\n{MEETING_TRANSCRIPT}",
        },
    ],
)

print(f"Type: {type(notes)}")
print(f"\nAttendees ({len(notes.attendees)}): {notes.attendees}")
print(f"\nDecisions ({len(notes.decisions)}):")
for d in notes.decisions:
    print(f"  - {d}")
print(f"\nAction items ({len(notes.action_items)}):")
for a in notes.action_items:
    print(f"  - {a}")
print(f"\nNext meeting: {notes.next_meeting}")

In [ ]:
# --- Extraction 2: Product Review ---

PRODUCT_REVIEW = """
I've been using this wireless keyboard for three months and have mixed feelings.
The build quality is excellent — solid, with satisfying tactile click. Battery life
is impressive: charged once, going strong for 6 weeks. However, the Bluetooth connection
drops at least twice a day, which is incredibly frustrating during video calls. The
software customization app only works on Windows — Mac users get default settings only.
For $120, I expected better reliability. I might keep it for the battery life and feel,
but I wouldn't recommend it to colleagues unless they're fine with connection issues.
"""

print("Extracting ProductReview...")
review = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=ProductReview,
    max_retries=3,
    messages=[
        {"role": "system", "content": "You are an expert at analyzing product reviews."},
        {"role": "user", "content": f"Analyze this review:\n\n{PRODUCT_REVIEW}"},
    ],
)

print(f"Sentiment:       {review.sentiment}")
print(f"Score:           {review.score}/5.0")
print(f"Would recommend: {review.would_recommend}")
print(f"Key features ({len(review.key_features)}):")
for f in review.key_features:
    print(f"  - {f}")

In [ ]:
# --- Extraction 3: Address from free text ---

ADDRESS_TEXT = """
Please ship my order to John Martinez at twelve forty-five West Oak Boulevard,
apartment three B, in San Francisco, California, nine four one zero two.
The buzzer code is 4521 but you can leave it at the front desk.
"""

print("Extracting UserAddress from text with numbers spelled out...")
address = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=UserAddress,
    max_retries=3,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an expert at parsing US mailing addresses from unstructured text. "
                "Always return a valid 5-digit ZIP code and 2-letter state code."
            ),
        },
        {"role": "user", "content": f"Extract the mailing address:\n\n{ADDRESS_TEXT}"},
    ],
)

print(f"Street:   {address.street}")
print(f"City:     {address.city}")
print(f"State:    {address.state}  (validator ran: uppercase 2-letter)")
print(f"ZIP:      {address.zip_code}  (validator ran: 5-digit)")
print(f"ZIP valid: {address.zip_code.isdigit() and len(address.zip_code) == 5}")

## Part 5 — Retry Mechanics: ValidationError → Reask → Fix

### What happens on validation failure

Instructor's retry loop:

```
for attempt in range(max_retries):
    response = openai_client.call(messages_with_schema)
    try:
        parsed = MyModel.model_validate_json(response.content)
        return parsed  # success
    except ValidationError as e:
        messages.append({
            "role": "user",
            "content": f"The response failed validation:\n{e}\nPlease try again."
        })
# After max_retries: raise InstructorRetryException
```

### Triggering a retry intentionally

To see the retry in action, define a model with a validator that fails on the first attempt.
We'll add a custom validator that requires the score to be a "round" number (1.0, 1.5, 2.0, etc.)
and watch instructor retry to get a conforming value.

In [ ]:
from pydantic import model_validator


class StrictProductReview(BaseModel):
    """ProductReview with an extra validator that requires score to be a multiple of 0.5."""

    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    score: float = Field(description="Score from 1.0 to 5.0, must be a multiple of 0.5", ge=1.0, le=5.0)

    @model_validator(mode="after")
    def score_must_be_half_point(self) -> "StrictProductReview":
        # Force score to nearest 0.5 — if the LLM gives 3.7, this raises
        if (self.score * 2) != int(self.score * 2):
            raise ValueError(
                f"Score must be a multiple of 0.5 (e.g. 1.0, 1.5, 2.0, ...). Got: {self.score}. "
                "Please round to the nearest 0.5."
            )
        return self


# This may trigger a retry if the LLM gives a non-half-point score
print("Extracting with StrictProductReview (score must be multiple of 0.5)...")
strict_review = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=StrictProductReview,
    max_retries=3,
    messages=[
        {"role": "system", "content": "You are an expert product reviewer."},
        {
            "role": "user",
            "content": (
                f"Analyze this review and give a score that is a multiple of 0.5:\n\n{PRODUCT_REVIEW}"
            ),
        },
    ],
)

print(f"Sentiment: {strict_review.sentiment}")
print(f"Score: {strict_review.score}  (is multiple of 0.5: {(strict_review.score * 2) == int(strict_review.score * 2)})")

## Part 6 — Field Validators: Enforcing Business Rules

### Why validators matter for extraction

Raw LLM output might give you:
- State as "California" instead of "CA"
- ZIP as "94102-1234" instead of "94102"
- Score as 5.7 (out of range)

Pydantic validators catch these before the data reaches your application.

### Validator types

| Type | When it runs | Use for |
|------|-------------|---------|
| `@field_validator` | After individual field parsing | Normalize or reject a single field |
| `@model_validator(mode="after")` | After all fields parsed | Cross-field validation |
| `Field(ge=1.0, le=5.0)` | During parsing | Simple range constraints |

In [ ]:
from pydantic import field_validator


class UserAddressV2(BaseModel):
    """Address model with comprehensive validators."""

    street: str = Field(description="Street number and name")
    city: str = Field(description="City name")
    state: str = Field(description="Two-letter US state code (e.g. CA, NY)")
    zip_code: str = Field(description="5-digit ZIP code (digits only)")

    # Normalize state: uppercase and strip whitespace
    @field_validator("state")
    @classmethod
    def normalize_state(cls, v: str) -> str:
        cleaned = v.upper().strip()
        # If model returns "California", extract standard abbreviations
        STATE_MAP = {
            "CALIFORNIA": "CA", "NEW YORK": "NY", "TEXAS": "TX",
            "FLORIDA": "FL", "ILLINOIS": "IL", "WASHINGTON": "WA",
        }
        return STATE_MAP.get(cleaned, cleaned[:2] if len(cleaned) > 2 else cleaned)

    # Extract and validate ZIP
    @field_validator("zip_code")
    @classmethod
    def validate_zip(cls, v: str) -> str:
        digits = "".join(c for c in v if c.isdigit())[:5]
        if len(digits) != 5:
            raise ValueError(f"ZIP code must have exactly 5 digits, extracted: {digits!r} from {v!r}")
        return digits

    # Capitalize city properly
    @field_validator("city")
    @classmethod
    def normalize_city(cls, v: str) -> str:
        return v.strip().title()


# Test validators
test_cases = [
    {"street": "1245 West Oak Blvd", "city": "san francisco", "state": "California", "zip_code": "94102-1234"},
    {"street": "800 N Michigan", "city": "CHICAGO", "state": "IL", "zip_code": "60611"},
]

for tc in test_cases:
    addr = UserAddressV2(**tc)
    print(f"Input:  state={tc['state']!r}, city={tc['city']!r}, zip={tc['zip_code']!r}")
    print(f"Output: state={addr.state!r}, city={addr.city!r}, zip={addr.zip_code!r}")
    print()

## Part 7 — Provider Portability

### The same code with Anthropic

`instructor.from_anthropic()` patches the Anthropic client with the same interface:

```python
import anthropic
import instructor

client = instructor.from_anthropic(anthropic.Anthropic())

notes = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    response_model=MeetingNotes,
    messages=[{"role": "user", "content": f"Extract notes from: {transcript}"}]
)
# returns a validated MeetingNotes — same as OpenAI version
```

Note: `client.messages.create` instead of `client.chat.completions.create` — Anthropic's API surface differs, but instructor abstracts the response parsing identically.

In [ ]:
# Demonstrate Anthropic provider (requires ANTHROPIC_API_KEY)

import os

anthropic_key = os.environ.get("ANTHROPIC_API_KEY", "")
if not anthropic_key:
    print("ANTHROPIC_API_KEY not set — skipping Anthropic demo.")
    print("Set it in .env or Colab Secrets to run this cell.")
else:
    import anthropic
    import instructor as instr

    ant_client = instr.from_anthropic(anthropic.Anthropic())

    ant_notes = ant_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        response_model=MeetingNotes,
        messages=[
            {
                "role": "user",
                "content": f"Extract structured notes from:\n\n{MEETING_TRANSCRIPT}",
            }
        ],
    )

    print("Anthropic extraction result:")
    print(f"Attendees: {ant_notes.attendees}")
    print(f"Decisions: {ant_notes.decisions}")
    print(f"Action items: {ant_notes.action_items}")
    print(f"\nType is same: {type(ant_notes) == MeetingNotes}")

## Exercises

### Exercise 1 — New extraction model

Define a `JobPosting` Pydantic model with fields:
- `title: str` — job title
- `company: str` — company name
- `required_skills: list[str]` — technical skills required
- `experience_years: int | None` — years of experience required (None if not specified)
- `remote: bool` — whether the role is remote

Extract from this text:
> "We're looking for a Senior Python Engineer at DataFlow Corp to join our ML platform team.
> You'll need 5+ years of experience with Python, FastAPI, and PostgreSQL. Docker and Kubernetes
> knowledge is a plus. This is a fully remote position."

---

### Exercise 2 — Nested models

Extend `MeetingNotes` to use a nested `ActionItem` model:

```python
class ActionItem(BaseModel):
    owner: str
    task: str
    deadline: str | None

class MeetingNotesV2(BaseModel):
    attendees: list[str]
    decisions: list[str]
    action_items: list[ActionItem]  # nested instead of list[str]
```

Re-extract from the same transcript. Does the model correctly identify owners and deadlines?

---

### Exercise 3 — max_retries=0 failure

Run the `UserAddress` extraction with `max_retries=0` on the ADDRESS_TEXT with numbers spelled out.
Does it succeed on the first attempt? If it fails, catch the exception and print the validation error.

In [ ]:
# ===== ANSWER KEY — Exercise 1: JobPosting =====

class JobPosting(BaseModel):
    title: str = Field(description="Job title")
    company: str = Field(description="Company name")
    required_skills: list[str] = Field(description="Technical skills explicitly required")
    experience_years: int | None = Field(
        default=None,
        description="Years of experience required, or null if not specified"
    )
    remote: bool = Field(description="True if the role is described as remote")


JOB_TEXT = """
We're looking for a Senior Python Engineer at DataFlow Corp to join our ML platform team.
You'll need 5+ years of experience with Python, FastAPI, and PostgreSQL.
Docker and Kubernetes knowledge is a plus. This is a fully remote position.
"""

job = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=JobPosting,
    max_retries=3,
    messages=[
        {"role": "system", "content": "You are an expert at parsing job postings."},
        {"role": "user", "content": f"Extract the job posting details:\n\n{JOB_TEXT}"},
    ],
)

print(f"Title:            {job.title}")
print(f"Company:          {job.company}")
print(f"Required skills:  {job.required_skills}")
print(f"Experience years: {job.experience_years}")
print(f"Remote:           {job.remote}")

In [ ]:
# ===== ANSWER KEY — Exercise 2: Nested ActionItem =====

class ActionItem(BaseModel):
    owner: str = Field(description="Person responsible for this action item")
    task: str = Field(description="Description of the task to be done")
    deadline: str | None = Field(
        default=None,
        description="Deadline date/time if mentioned, else null"
    )


class MeetingNotesV2(BaseModel):
    attendees: list[str] = Field(description="Names of all meeting attendees")
    decisions: list[str] = Field(description="Decisions made during the meeting")
    action_items: list[ActionItem] = Field(
        description="Structured action items with owner, task, and optional deadline"
    )


notes_v2 = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=MeetingNotesV2,
    max_retries=3,
    messages=[
        {"role": "system", "content": "Extract structured meeting notes with nested action items."},
        {"role": "user", "content": f"Extract from:\n\n{MEETING_TRANSCRIPT}"},
    ],
)

print(f"Attendees: {notes_v2.attendees}")
print(f"\nStructured action items ({len(notes_v2.action_items)}):")
for ai in notes_v2.action_items:
    deadline_str = f" by {ai.deadline}" if ai.deadline else ""
    print(f"  [{ai.owner}] {ai.task}{deadline_str}")

In [ ]:
# ===== ANSWER KEY — Exercise 3: max_retries=0 failure =====

from instructor.exceptions import InstructorRetryException

try:
    addr_zero_retry = client.chat.completions.create(
        model="gpt-4o-mini",
        response_model=UserAddress,
        max_retries=0,  # no retry
        messages=[
            {"role": "system", "content": "Extract the address. State must be 2-letter code. ZIP must be 5 digits."},
            {"role": "user", "content": f"Extract address from:\n\n{ADDRESS_TEXT}"},
        ],
    )
    print(f"Success on first attempt: {addr_zero_retry.street}, {addr_zero_retry.city}, {addr_zero_retry.state} {addr_zero_retry.zip_code}")
    print("(The model produced valid output without retry.)")

except InstructorRetryException as e:
    print(f"InstructorRetryException raised after 1 attempt:")
    print(f"  {str(e)[:300]}")
    print("\nConclusion: max_retries=0 means raise immediately on first validation error.")
except Exception as e:
    print(f"Other error: {type(e).__name__}: {e}")

## Workshop Complete

You have mastered instructor's core pattern:

- **`instructor.from_openai()`** — patches the OpenAI client with `response_model` support
- **`response_model=MyModel`** — LLM output is validated against your Pydantic schema
- **`max_retries=N`** — on `ValidationError`, instructor reasks with the error message
- **`@field_validator`** — enforce business rules (state codes, ZIP digits, score ranges)
- **Provider portability** — `instructor.from_anthropic()` works identically

**Key insight**: instructor is not a new framework — it's a thin patch on any OpenAI-compatible client. The result is that `response_model=` turns unstructured text into type-safe Python objects with automatic recovery from model errors.

---

Next: **example 118** — Mirascope: Pydantic-first LLM calls with provider switching.

---
### Further reading
- [instructor docs](https://python.useinstructor.com/)
- [instructor GitHub](https://github.com/jxnl/instructor)
- [Pydantic validators](https://docs.pydantic.dev/latest/concepts/validators/)
- [instructor with Anthropic](https://python.useinstructor.com/integrations/anthropic/)</cell id="cell-md-023"></cell>
